# KrishiVerify: prototype for detecting suspicious fertiliser and pesticide packs

This notebook is a working prototype that covers your **Objective 1 and 2 implementation steps** and sketches the technical side of Objectives 3 and 4. Every part runs on synthetic (simulated) data, so you can run it top to bottom in Google Colab with no uploads.

**What it can and cannot claim.** A QR code and a signature prove that a pack carries a *code issued by a registered manufacturer*. They cannot prove what is chemically inside the bag. So KrishiVerify never says "authentic". It says **"consistent with manufacturer records"**, and uses seal checks, a scratch code, scan history and an optional test strip as extra evidence.

| Objective | Where in this notebook |
|---|---|
| 1. Traceability + authentication, field validation | Parts 1 to 6, Part 9 (QR decoding under bad conditions), `field_trial_template.csv` |
| 2. Scalability, reliability, cost | Parts 7 and 8 |
| 3. Interoperable governance | `Registry` class here, full model in the design document |
| 4. User-centred, secure, offline workflow | Offline verifier, 3 languages, attack scenarios (Part 4) |

**How to read the code:** each Part starts with a plain-English note on *what* and *why*. Run the cells in order.

In [5]:
# Part 0: setup. Colab already has everything except `cryptography` being up to date.
%pip install numpy pandas opencv-python-headless scikit-learn cryptography

   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
    --------------------------------------- 0.3/12.7 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.7 MB 2.1 MB/s eta 0:00:06
   ---- ----------------------------------- 1.3/12.7 MB 2.5 MB/s eta 0:00:05
   ----- ---------------------------------- 1.8/12.7 MB 2.3 MB/s eta 0:00:05
   ------ --------------------------------- 2.1/12.7 MB 2.2 MB/s eta 0:00:05
   ------ --------------------------------- 2.1/12.7 MB 2.2 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/12.7 MB 1.6 MB/s eta 0:00:07
   --------- ------------------------------ 2.9/12.7 MB 1.7 MB/s eta 0:00:06
   ---------- ----------------------------- 3.4/12.7 MB 1.8 MB/s eta 0:00:06
   ------------ --------------------------- 3.9/12.7 MB 1.9 MB/s eta 0:00:05
   -------------- ------------------------- 4.5/12.7 MB 1.9 MB/s eta 0:00:05
   --------------- ------------------------ 5.0/12.7 MB 2.0 MB/s eta 0:00:04
   ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import base64, json, hashlib, time, sqlite3, os, random, secrets, gzip, math, statistics, tempfile
import datetime as dt
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey, Ed25519PublicKey
from cryptography.hazmat.primitives import serialization
from cryptography.exceptions import InvalidSignature

random.seed(42)
np.random.seed(42)
OUT = "krishiverify_outputs"
os.makedirs(OUT, exist_ok=True)

def b64e(b: bytes) -> str:
    return base64.urlsafe_b64encode(b).rstrip(b"=").decode()

def b64d(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))

def sha256(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()

## Part 1: Map the supply chain and its stakeholders

**What:** list who touches a product and what each person is *allowed* to record.
**Why:** traceability only works if the system knows that, for example, a retailer cannot "manufacture" a bag, and a bag cannot be "sold" before it was "received".

Supply chain: **Manufacturer → Distributor → Retailer → Farmer (end user)**.
Recorded events: `MANUFACTURED → SHIPPED → RECEIVED → (SHIPPED → RECEIVED) → SOLD`.
Farmers do not sign anything. They only *verify* (scan), and each scan is logged.

Each signing actor gets an **Ed25519 key pair**. The private key stays with the actor. The public key goes into a **Registry**, which stands in for the regulator-run key register (Objective 3).

In [7]:
@dataclass
class Actor:
    actor_id: str
    role: str          # manufacturer | distributor | retailer
    name: str
    region: str
    key: Ed25519PrivateKey = field(default_factory=Ed25519PrivateKey.generate, repr=False)

    @property
    def pub(self) -> bytes:
        return self.key.public_key().public_bytes(serialization.Encoding.Raw, serialization.PublicFormat.Raw)

    def sign(self, data: bytes) -> bytes:
        return self.key.sign(data)

# Which event types may follow which, and which roles may record them.
VALID_NEXT = {None: {"MANUFACTURED"}, "MANUFACTURED": {"SHIPPED"}, "SHIPPED": {"RECEIVED"},
              "RECEIVED": {"SHIPPED", "SOLD"}, "SOLD": set()}
ROLE_CAN = {"MANUFACTURED": {"manufacturer"}, "SHIPPED": {"manufacturer", "distributor"},
            "RECEIVED": {"distributor", "retailer"}, "SOLD": {"retailer"}}

class Registry:
    """Stands in for a regulator / sector-body register of approved actors and recalled batches."""
    def __init__(self):
        self.actors, self.revoked_batches = {}, set()

    def register(self, a: Actor):
        self.actors[a.actor_id] = {"role": a.role, "name": a.name, "region": a.region,
                                   "pub": a.pub.hex(), "status": "active"}

    def revoke_actor(self, actor_id): self.actors[actor_id]["status"] = "revoked"
    def recall_batch(self, mfr_id, batch): self.revoked_batches.add(f"{mfr_id}:{batch}")

    def snapshot(self) -> dict:
        """The small file a phone downloads when online and keeps for offline checks."""
        return {"as_of": dt.date.today().isoformat(),
                "actors": {k: {"pub": v["pub"], "role": v["role"], "status": v["status"]} for k, v in self.actors.items()},
                "revoked_batches": sorted(self.revoked_batches)}

registry = Registry()
mfr   = Actor("MFR-01",  "manufacturer", "GreenGrow Fertilisers (demo)", "Telangana")
dist  = Actor("DIST-01", "distributor",  "Deccan Agro Distributors (demo)", "Telangana")
ret1  = Actor("RET-01",  "retailer",     "Rythu Seva Kendra shop (demo)", "Telangana")
ret2  = Actor("RET-02",  "retailer",     "Village agri store (demo)", "Andhra Pradesh")
for a in (mfr, dist, ret1, ret2):
    registry.register(a)

print("Registered actors:", list(registry.actors))
print("Snapshot size the phone must cache:", len(json.dumps(registry.snapshot())), "bytes")

Registered actors: ['MFR-01', 'DIST-01', 'RET-01', 'RET-02']
Snapshot size the phone must cache: 574 bytes


## Part 2: Tamper-evident traceability backend

**What:** a database table where every event is (a) **signed** by the actor who recorded it and (b) **chained** to the previous event by a hash, like a mini blockchain without the network overhead.
**Why:** if anyone edits or deletes an old row, the chain breaks and `verify_chain()` shows exactly where. If an attacker re-computes the hashes after editing, they still cannot forge the signatures without the actors' private keys.

The same class can run with `secure=False`, which skips hashing and signing. That is our **conventional database baseline** for the comparison in Part 7.

In [8]:
class Rejected(Exception):
    pass

GENESIS = "0" * 64

class Ledger:
    def __init__(self, registry: Registry, path=":memory:", secure=True):
        self.registry, self.secure, self.path = registry, secure, path
        self.db = sqlite3.connect(path, isolation_level=None)   # we control transactions ourselves
        self.db.execute("PRAGMA journal_mode=WAL")
        self.db.execute("PRAGMA synchronous=NORMAL")
        self.db.executescript("""
        CREATE TABLE IF NOT EXISTS events(
            seq INTEGER PRIMARY KEY AUTOINCREMENT, event_id TEXT UNIQUE, unit_id TEXT, type TEXT,
            actor_id TEXT, ts TEXT, body TEXT, prev_hash TEXT, hash TEXT, sig TEXT);
        CREATE INDEX IF NOT EXISTS ix_unit ON events(unit_id);
        CREATE TABLE IF NOT EXISTS scans(unit_id TEXT, ts TEXT, region TEXT);
        CREATE INDEX IF NOT EXISTS ix_scan ON scans(unit_id);
        CREATE TABLE IF NOT EXISTS attempts(unit_id TEXT PRIMARY KEY, wrong INTEGER);
        """)

    @staticmethod
    def _canon(event_id, unit_id, etype, actor_id, ts, body, prev_hash):
        return f"{event_id}|{unit_id}|{etype}|{actor_id}|{ts}|{body}|{prev_hash}"

    def append(self, unit_id, etype, actor: Actor, body=None, event_id=None, ts=None, _crash=False):
        event_id = event_id or secrets.token_hex(8)
        c = self.db
        dup = c.execute("SELECT seq FROM events WHERE event_id=?", (event_id,)).fetchone()
        if dup:                                   # idempotent: a re-sent request changes nothing
            return "DUPLICATE", dup[0]
        reg = self.registry.actors.get(actor.actor_id)
        if not reg or reg["status"] != "active":
            raise Rejected(f"{actor.actor_id} is not an active registered actor")
        if reg["role"] not in ROLE_CAN[etype]:
            raise Rejected(f"role {reg['role']} may not record {etype}")
        last = c.execute("SELECT type FROM events WHERE unit_id=? ORDER BY seq DESC LIMIT 1", (unit_id,)).fetchone()
        last_t = last[0] if last else None
        if etype not in VALID_NEXT[last_t]:
            raise Rejected(f"{etype} cannot follow {last_t}")
        ts = ts or dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
        body_json = json.dumps(body or {}, separators=(",", ":"), sort_keys=True)
        c.execute("BEGIN")
        try:
            if self.secure:
                prev = c.execute("SELECT hash FROM events ORDER BY seq DESC LIMIT 1").fetchone()
                prev_hash = prev[0] if prev else GENESIS
                h = sha256(self._canon(event_id, unit_id, etype, actor.actor_id, ts, body_json, prev_hash))
                sig = actor.sign(h.encode()).hex()
            else:
                prev_hash = h = sig = ""
            cur = c.execute("INSERT INTO events(event_id,unit_id,type,actor_id,ts,body,prev_hash,hash,sig) VALUES(?,?,?,?,?,?,?,?,?)",
                            (event_id, unit_id, etype, actor.actor_id, ts, body_json, prev_hash, h, sig))
            if _crash:
                raise RuntimeError("simulated device failure mid-write")
            c.execute("COMMIT")
            return "OK", cur.lastrowid
        except Exception:
            c.execute("ROLLBACK")
            raise

    # ---- integrity -------------------------------------------------------------------------
    def verify_chain(self):
        """Walk every event. Returns (ok, first_bad_seq, reason)."""
        prev = GENESIS
        rows = self.db.execute("SELECT seq,event_id,unit_id,type,actor_id,ts,body,prev_hash,hash,sig FROM events ORDER BY seq")
        for seq, eid, uid, et, aid, ts, body, ph, h, sig in rows:
            if ph != prev:
                return False, seq, "broken link (a row was removed or reordered)"
            if sha256(self._canon(eid, uid, et, aid, ts, body, ph)) != h:
                return False, seq, "content altered"
            pub = Ed25519PublicKey.from_public_bytes(bytes.fromhex(self.registry.actors[aid]["pub"]))
            try:
                pub.verify(bytes.fromhex(sig), h.encode())
            except InvalidSignature:
                return False, seq, "signature invalid (hashes were recomputed by someone without the key)"
            prev = h
        return True, None, "ok"

    def anchor(self):
        """Head hash + row count. Publish this somewhere outside the database (printed report, regulator,
        public timestamp). It is what detects deletion of the NEWEST rows."""
        n = self.db.execute("SELECT COUNT(*) FROM events").fetchone()[0]
        h = self.db.execute("SELECT hash FROM events ORDER BY seq DESC LIMIT 1").fetchone()
        return {"count": n, "head": h[0] if h else GENESIS}

    def check_anchor(self, anchor):
        return self.anchor() == anchor

    # ---- per-unit reads used by the verifier -------------------------------------------------
    def unit_history(self, unit_id):
        rows = self.db.execute("SELECT event_id,type,actor_id,ts,body,prev_hash,hash,sig FROM events WHERE unit_id=? ORDER BY seq",
                               (unit_id,)).fetchall()
        ok = True
        if self.secure:
            for eid, et, aid, ts, body, ph, h, sig in rows:
                if sha256(self._canon(eid, unit_id, et, aid, ts, body, ph)) != h:
                    ok = False
                pub = Ed25519PublicKey.from_public_bytes(bytes.fromhex(self.registry.actors[aid]["pub"]))
                try:
                    pub.verify(bytes.fromhex(sig), h.encode())
                except InvalidSignature:
                    ok = False
        return [dict(type=r[1], actor=r[2], ts=r[3], body=json.loads(r[4])) for r in rows], ok

    def record_scan(self, unit_id, region):
        self.db.execute("INSERT INTO scans VALUES(?,?,?)", (unit_id, dt.datetime.now(dt.timezone.utc).isoformat(), region))

    def scan_stats(self, unit_id):
        n, r = self.db.execute("SELECT COUNT(*), COUNT(DISTINCT region) FROM scans WHERE unit_id=?", (unit_id,)).fetchone()
        return n, r

    def check_scratch(self, unit_id, scratch, manufactured_body):
        """Returns 'ok' | 'wrong' | 'locked'. Five wrong guesses lock the unit (stops brute force)."""
        w = self.db.execute("SELECT wrong FROM attempts WHERE unit_id=?", (unit_id,)).fetchone()
        if w and w[0] >= 5:
            return "locked"
        if sha256(unit_id + scratch) == manufactured_body.get("scratch_hash"):
            return "ok"
        self.db.execute("INSERT INTO attempts VALUES(?,1) ON CONFLICT(unit_id) DO UPDATE SET wrong=wrong+1", (unit_id,))
        return "wrong"

ledger = Ledger(registry)
print("Ledger created. Chain on an empty ledger:", ledger.verify_chain())

Ledger created. Chain on an empty ledger: (True, None, 'ok')


## Part 3: Unique pack identity (signed QR + hidden scratch code)

**What:** each pack gets a QR code that contains a short JSON payload (manufacturer, SKU, batch, serial, expiry) plus an **Ed25519 signature** over that payload.
**Why:** anyone can print a QR code, but only the manufacturer can produce a *valid signature*. A phone that holds the manufacturer's public key can check it **with no internet**.

**Honest weakness, and the fix.** A signature stops *forged* codes, but not *copied* ones: a counterfeiter can photograph a real QR and print it on fake bags. Two defences are built in:
1. **Scratch code**: a short secret printed under a scratch panel. Only its hash is stored in the ledger. A copied QR does not carry the secret.
2. **Scan history**: the same unit scanned many times, or from several regions, is a clone warning.

In [9]:
def issue_unit(mfr: Actor, ledger: Ledger, sku: str, batch: str, serial: int, mfg_date: str, exp_date: str):
    unit_id = f"{mfr.actor_id}:{batch}:{serial:06d}"
    payload = {"v": 1, "m": mfr.actor_id, "s": sku, "b": batch, "n": serial, "x": exp_date}
    pbytes = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode()
    qr_text = b64e(pbytes) + "." + b64e(mfr.sign(pbytes))
    scratch = secrets.token_hex(4)                       # printed under the scratch panel
    ledger.append(unit_id, "MANUFACTURED", mfr,
                  {"sku": sku, "batch": batch, "mfg": mfg_date, "scratch_hash": sha256(unit_id + scratch)})
    return unit_id, qr_text, scratch

def move_unit(unit_id, ledger, mfr, dist, retailer, sell=True):
    ledger.append(unit_id, "SHIPPED", mfr, {"to": dist.actor_id})
    ledger.append(unit_id, "RECEIVED", dist, {})
    ledger.append(unit_id, "SHIPPED", dist, {"to": retailer.actor_id})
    ledger.append(unit_id, "RECEIVED", retailer, {})
    if sell:
        ledger.append(unit_id, "SOLD", retailer, {})

uid, qr, scratch = issue_unit(mfr, ledger, "UREA-46-50KG", "B2610A", 1, "2026-10-01", "2028-10-01")
move_unit(uid, ledger, mfr, dist, ret1)
print("Unit id     :", uid)
print("QR text     :", qr)
print("QR length   :", len(qr), "characters")
print("Scratch code:", scratch, "(printed on the pack, hash only stored in ledger)")
print("Chain valid :", ledger.verify_chain())

Unit id     : MFR-01:B2610A:000001
QR text     : eyJiIjoiQjI2MTBBIiwibSI6Ik1GUi0wMSIsIm4iOjEsInMiOiJVUkVBLTQ2LTUwS0ciLCJ2IjoxLCJ4IjoiMjAyOC0xMC0wMSJ9.cPddvCjWi27eNVHg3oo-4O2C2uMFRF-WkrX7JoaK_yvDjIQ70I0Qrsopycx_AoUIy55zqXtSAEhD8ZGXKB7LCw
QR length   : 187 characters
Scratch code: 1d386de6 (printed on the pack, hash only stored in ledger)
Chain valid : (True, None, 'ok')


## Part 4: Verification engine (rules) and farmer-facing message

**What:** the app logic. It runs in two modes:
- **Offline** (no network): checks signature, whether the manufacturer's key is trusted, recall list, expiry, seal.
- **Online**: all of the above plus the ledger (was this unit ever registered? sold through a registered retailer? is the history intact?), scan history, scratch code.

Checks are scored. **Hard failures** (bad signature, unknown manufacturer, recalled batch) give `INVALID` straight away. Everything else adds risk points, so an *informal* supply chain (e.g. an unregistered shop) gives a caution, not a false alarm.

The optional **test strip** is just a colour comparison: the farmer photographs the strip next to a white reference card, and we measure how far its colour is from the expected colour. It is weak evidence by design and is never enough alone.

Farmer messages are in **English, Hindi and Telugu** (please have a native speaker review the wording before any field trial).

In [10]:
MESSAGES = {
 "CONSISTENT": {"en": "GREEN: Code matches manufacturer records. Still check the seal and the bag.",
                "hi": "हरा: कोड निर्माता के रिकॉर्ड से मेल खाता है। फिर भी सील और बैग जाँचें।",
                "te": "ఆకుపచ్చ: కోడ్ తయారీదారు రికార్డులతో సరిపోతోంది. అయినా సీల్, బ్యాగ్ చూడండి."},
 "UNVERIFIED_OFFLINE": {"en": "YELLOW: Code is genuine-looking but could not be checked online. Check again when you have network.",
                "hi": "पीला: कोड सही लगता है पर ऑनलाइन जाँच नहीं हुई। नेटवर्क मिलने पर फिर जाँचें।",
                "te": "పసుపు: కోడ్ సరిగ్గా ఉంది కానీ ఆన్‌లైన్ తనిఖీ కాలేదు. నెట్‌వర్క్ వచ్చాక మళ్లీ చూడండి."},
 "CAUTION": {"en": "YELLOW: Some checks look unusual. Ask the shop for the bill and report if unsure.",
                "hi": "पीला: कुछ जाँचें असामान्य हैं। दुकान से बिल लें और शक हो तो शिकायत करें।",
                "te": "పసుపు: కొన్ని తనిఖీలు అసాధారణంగా ఉన్నాయి. దుకాణం నుండి బిల్లు తీసుకోండి, అనుమానం ఉంటే ఫిర్యాదు చేయండి."},
 "SUSPICIOUS": {"en": "RED: Do not use. This pack looks suspicious. Keep the bag and report it.",
                "hi": "लाल: उपयोग न करें। यह पैक संदिग्ध है। बैग रखें और शिकायत करें।",
                "te": "ఎరుపు: వాడవద్దు. ఈ ప్యాక్ అనుమానాస్పదంగా ఉంది. బ్యాగ్ ఉంచి ఫిర్యాదు చేయండి."},
 "INVALID": {"en": "RED: Invalid or recalled code. Do not use. Report it.",
                "hi": "लाल: अमान्य या वापस मँगाया गया कोड। उपयोग न करें। शिकायत करें।",
                "te": "ఎరుపు: చెల్లని లేదా రీకాల్ చేసిన కోడ్. వాడవద్దు. ఫిర్యాదు చేయండి."},
}

# Expected strip colours are PLACEHOLDERS. Calibrate with real strips and lab-tested samples before use.
STRIP_REFERENCE = {"UREA-46-50KG": (120, 190, 90)}

def strip_deviation(measured_rgb, expected_rgb):
    """0 = identical colour, 1 = as far apart as possible (simple RGB distance)."""
    return float(np.linalg.norm(np.array(measured_rgb) - np.array(expected_rgb)) / (255 * math.sqrt(3)))

def parse_qr(qr):
    try:
        p, s = qr.split(".")
        pb, sig = b64d(p), b64d(s)
        return pb, sig, json.loads(pb)
    except Exception:
        return None

class Verifier:
    """What runs on the farmer's phone / Raspberry Pi kiosk."""
    def __init__(self, cache: dict, ledger: Ledger = None):
        self.cache, self.ledger = cache, ledger

    def verify(self, qr, today=None, scratch=None, seal="unknown", strip_rgb=None, region="Telangana", online=True):
        today = today or dt.date.today().isoformat()
        flags, score = [], 0
        parsed = parse_qr(qr)
        if not parsed:
            return self._out("INVALID", 100, ["unreadable_code"], online)
        pb, sig, p = parsed

        # ---- offline checks (need only the cached registry snapshot) ----
        actor = self.cache["actors"].get(p.get("m"))
        if not actor or actor["role"] != "manufacturer":
            return self._out("INVALID", 100, ["unknown_manufacturer"], online)
        if actor["status"] != "active":
            return self._out("INVALID", 100, ["manufacturer_revoked"], online)
        try:
            Ed25519PublicKey.from_public_bytes(bytes.fromhex(actor["pub"])).verify(sig, pb)
        except InvalidSignature:
            return self._out("INVALID", 100, ["bad_signature"], online)
        if f"{p['m']}:{p['b']}" in self.cache["revoked_batches"]:
            return self._out("INVALID", 100, ["batch_recalled"], online)
        if p["x"] < today:
            flags.append("expired"); score += 60
        if seal == "broken":
            flags.append("seal_broken"); score += 40
        if strip_rgb is not None and p["s"] in STRIP_REFERENCE:
            dev = strip_deviation(strip_rgb, STRIP_REFERENCE[p["s"]])
            if dev > 0.20:
                flags.append(f"strip_colour_off({dev:.2f})"); score += 30

        # ---- online checks ----
        if not online or self.ledger is None:
            verdict = "SUSPICIOUS" if score >= 50 else "CAUTION" if score >= 25 else "UNVERIFIED_OFFLINE"
            return self._out(verdict, score, flags, online=False)

        uid = f"{p['m']}:{p['b']}:{p['n']:06d}"
        events, hist_ok = self.ledger.unit_history(uid)
        if not events:
            return self._out("INVALID", 100, flags + ["unit_never_registered"], True)
        if not hist_ok:
            flags.append("ledger_history_tampered"); score += 60
        if events[-1]["type"] != "SOLD":
            flags.append("no_registered_sale"); score += 25          # informal chain: caution, not rejection
        self.ledger.record_scan(uid, region)
        n, regions = self.ledger.scan_stats(uid)
        # Weights are hand-set starting points. A genuine bag is rarely scanned >3 times or from 3+ regions.
        if n > 3:
            flags.append(f"many_scans({n})"); score += 25
        if regions >= 3:
            flags.append(f"scanned_in_{regions}_regions"); score += 30
        if scratch:
            res = self.ledger.check_scratch(uid, scratch, events[0]["body"])
            if res == "ok":
                score = max(0, score - 15)
            else:
                flags.append(f"scratch_code_{res}"); score += 50
        verdict = "SUSPICIOUS" if score >= 50 else "CAUTION" if score >= 25 else "CONSISTENT"
        return self._out(verdict, score, flags, True)

    @staticmethod
    def _out(verdict, score, flags, online):
        return {"verdict": verdict, "risk": min(score, 100), "flags": flags, "online": online}

def farmer_view(result, lang="en"):
    return MESSAGES[result["verdict"]][lang]

phone = Verifier(registry.snapshot(), ledger)
r = phone.verify(qr, scratch=scratch, seal="intact")
print(r)
for lang in ("en", "hi", "te"):
    print(lang, "->", farmer_view(r, lang))

{'verdict': 'CONSISTENT', 'risk': 0, 'flags': [], 'online': True}
en -> GREEN: Code matches manufacturer records. Still check the seal and the bag.
hi -> हरा: कोड निर्माता के रिकॉर्ड से मेल खाता है। फिर भी सील और बैग जाँचें।
te -> ఆకుపచ్చ: కోడ్ తయారీదారు రికార్డులతో సరిపోతోంది. అయినా సీల్, బ్యాగ్ చూడండి.


## Part 5: Attack scenarios (this becomes your threat-model validation table)

**What:** we play the counterfeiter and see what the system says.
**Why:** a security claim is only believable if you have tried to break it. Each row below is one attack from the threat model. The last two rows attack the *database itself*.

In [11]:
def fresh_world():
    reg = Registry()
    for a in (mfr, dist, ret1, ret2):
        reg.register(a)
    led = Ledger(reg)
    return reg, led

reg, led = fresh_world()
attacker = Actor("MFR-01", "manufacturer", "Attacker pretending to be GreenGrow", "Unknown")   # same id, different key

rows = []
def run(name, expected, qr_text, scratch=None, seal="intact", strip=None, region="Telangana", online=True, v=None, pre=None):
    v = v or Verifier(reg.snapshot(), led)
    if pre: pre()
    res = v.verify(qr_text, scratch=scratch, seal=seal, strip_rgb=strip, region=region, online=online)
    rows.append({"attack / case": name, "expected": expected, "got": res["verdict"], "flags": ", ".join(res["flags"]) or "-",
                 "pass": "yes" if res["verdict"] == expected else "NO"})

# A. genuine, sold through a registered shop
u, q, s = issue_unit(mfr, led, "UREA-46-50KG", "B2610A", 10, "2026-10-01", "2028-10-01"); move_unit(u, led, mfr, dist, ret1)
run("A. Genuine pack, correct scratch code", "CONSISTENT", q, scratch=s)

# B. forged QR signed with the attacker's own key
u2, q2, s2 = issue_unit(mfr, led, "UREA-46-50KG", "B2610A", 11, "2026-10-01", "2028-10-01")   # real, so we can copy its payload
forged_payload = json.dumps({"v":1,"m":"MFR-01","s":"UREA-46-50KG","b":"B2610A","n":9999,"x":"2028-10-01"}, separators=(",",":"), sort_keys=True).encode()
forged_qr = b64e(forged_payload) + "." + b64e(attacker.sign(forged_payload))
run("B. Forged QR signed with attacker's key", "INVALID", forged_qr)

# C. real QR with the batch edited (e.g. to hide a recall)
p_b64, sig_b64 = q.split(".")
edited = json.loads(b64d(p_b64)); edited["b"] = "B2699Z"
edited_qr = b64e(json.dumps(edited, separators=(",",":"), sort_keys=True).encode()) + "." + sig_b64
run("C. Real QR, payload edited after signing", "INVALID", edited_qr)

# D. cloned QR printed on a fake bag, scratch panel has a made-up code
run("D. Cloned QR, fake scratch code", "SUSPICIOUS", q, scratch="deadbeef")

# E. cloned QR scanned over and over from 3 regions (no scratch code entered)
u3, q3, s3 = issue_unit(mfr, led, "UREA-46-50KG", "B2610A", 12, "2026-10-01", "2028-10-01"); move_unit(u3, led, mfr, dist, ret1)
for rg in ["Telangana", "Odisha", "Punjab", "Odisha", "Telangana"]:
    Verifier(reg.snapshot(), led).verify(q3, region=rg)
run("E. Same QR scanned 6x across 3 regions", "SUSPICIOUS", q3, region="Bihar")

# F. recalled batch
reg.recall_batch("MFR-01", "B2611R")
u4, q4, s4 = issue_unit(mfr, led, "UREA-46-50KG", "B2611R", 1, "2026-10-02", "2028-10-02")
run("F. Recalled batch (phone has refreshed snapshot)", "INVALID", q4, v=Verifier(reg.snapshot(), led))

# G. valid unit never sold through a registered shop (informal chain)
u5, q5, s5 = issue_unit(mfr, led, "UREA-46-50KG", "B2610A", 13, "2026-10-01", "2028-10-01")
run("G. Valid unit, no registered sale (informal chain)", "CAUTION", q5)

# H. valid code, broken seal and off-colour strip on a repackaged bag
run("H. Broken seal + wrong strip colour", "SUSPICIOUS", q, scratch=s, seal="broken", strip=(200, 120, 60))

# I. offline check of a genuine code
run("I. Genuine code, phone offline", "UNVERIFIED_OFFLINE", q, online=False)

# J. offline check of a forged code
run("J. Forged code, phone offline", "INVALID", forged_qr, online=False)

# K. someone edits an old row in the database
led.db.execute("UPDATE events SET body=? WHERE seq=2", ('{"to":"EVIL-99"}',))
ok, seq, why = led.verify_chain()
rows.append({"attack / case": "K. Edit an old ledger row", "expected": "detected", "got": "detected" if not ok else "MISSED",
             "flags": f"seq {seq}: {why}", "pass": "yes" if not ok else "NO"})

# L. truncate the newest rows, then compare to a published anchor
reg2, led2 = fresh_world()
for i in range(5):
    uu, _, _ = issue_unit(mfr, led2, "UREA-46-50KG", "B1", i, "2026-10-01", "2028-10-01")
anchor = led2.anchor()
led2.db.execute("DELETE FROM events WHERE seq > 3")
chain_ok = led2.verify_chain()[0]
anchor_ok = led2.check_anchor(anchor)
rows.append({"attack / case": "L. Delete the newest rows", "expected": "detected via anchor", "got": "detected via anchor" if not anchor_ok else "MISSED",
             "flags": f"hash chain alone says ok={chain_ok}; anchor mismatch={not anchor_ok}", "pass": "yes" if (chain_ok and not anchor_ok) else "NO"})

attack_table = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80, "display.width", 200)
attack_table.to_csv(f"{OUT}/attack_validation.csv", index=False)
attack_table

,attack / case,expected,got,flags,pass
0,"A. Genuine pack, correct scratch code",CONSISTENT,CONSISTENT,-,yes
1,B. Forged QR signed with attacker's key,INVALID,INVALID,bad_signature,yes
2,"C. Real QR, payload edited after signing",INVALID,INVALID,bad_signature,yes
3,"D. Cloned QR, fake scratch code",SUSPICIOUS,SUSPICIOUS,scratch_code_wrong,yes
4,E. Same QR scanned 6x across 3 regions,SUSPICIOUS,SUSPICIOUS,"many_scans(6), scanned_in_4_regions",yes
5,F. Recalled batch (phone has refreshed snapshot),INVALID,INVALID,batch_recalled,yes
6,"G. Valid unit, no registered sale (informal chain)",CAUTION,CAUTION,no_registered_sale,yes
7,H. Broken seal + wrong strip colour,SUSPICIOUS,SUSPICIOUS,"seal_broken, strip_colour_off(0.25)",yes
8,"I. Genuine code, phone offline",UNVERIFIED_OFFLINE,UNVERIFIED_OFFLINE,-,yes
9,"J. Forged code, phone offline",INVALID,INVALID,bad_signature,yes


Case **L** is deliberate: a hash chain on its own cannot notice that the *newest* rows were deleted, because nothing comes after them. Publishing the `anchor` (head hash + row count) outside the database, for example to the regulator each day, closes that gap. This is the kind of limitation your threat model should state openly.

## Part 6: Machine-learning risk model (Random Forest vs Gradient Boosting)

**What:** the rules in Part 4 use fixed points. A model can learn how the signals combine, especially for hard cases like a *real QR copied onto a diluted product*, where the code looks perfect and only the price, seal and strip are odd.

**Important honesty note.** I have not seen your `counterfeit_products_clean.csv`, so this cell trains on **simulated scan records** whose overlap between genuine and fake is set by me. The numbers show that the *pipeline works*; they are **not evidence of real-world accuracy**. The cell after this one lets you swap in your Kaggle file.

**How to read the metrics (beginner notes):**
- **Precision**: of the packs the model flagged, how many were really fake.
- **Recall**: of all fake packs, how many the model caught. For farmers, a missed fake usually costs more than a false alarm, so watch recall.
- **F1**: one number balancing the two. **ROC AUC**: how well it ranks fake above genuine regardless of threshold (0.5 = coin flip, 1.0 = perfect).
- We split the data into **train** (the model learns) and **test** (the model is graded on rows it never saw).

In [12]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

def simulate_scans(n=4000, fake_rate=0.15, seed=7):
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n):
        if rng.random() > fake_rate:                                   # ---- genuine scan
            kind = "genuine"
            r = dict(sig_valid=1, scan_count=1 + rng.poisson(0.8), regions=1 + (rng.random() < 0.04),
                     registered_sale=int(rng.random() < 0.88), scratch_wrong=int(rng.random() < 0.01),
                     scratch_ok=int(rng.random() < 0.55), seal_broken=int(rng.random() < 0.03),
                     strip_dev=abs(rng.normal(0.07, 0.05)), price_ratio=rng.normal(1.0, 0.06))
        else:
            kind = rng.choice(["cloned_qr", "diluted_repack", "forged_qr", "diverted"], p=[.3, .3, .2, .2])
            if kind == "cloned_qr":
                r = dict(sig_valid=1, scan_count=4 + rng.poisson(4), regions=int(rng.integers(2, 6)),
                         registered_sale=int(rng.random() < 0.3), scratch_wrong=int(rng.random() < 0.6),
                         scratch_ok=int(rng.random() < 0.1), seal_broken=int(rng.random() < 0.3),
                         strip_dev=abs(rng.normal(0.2, 0.1)), price_ratio=rng.uniform(0.7, 0.95))
            elif kind == "diluted_repack":      # hardest: code looks genuine and was scanned like a genuine one
                r = dict(sig_valid=1, scan_count=1 + rng.poisson(1.0), regions=1,
                         registered_sale=int(rng.random() < 0.6), scratch_wrong=int(rng.random() < 0.05),
                         scratch_ok=int(rng.random() < 0.3), seal_broken=int(rng.random() < 0.5),
                         strip_dev=abs(rng.normal(0.28, 0.12)), price_ratio=rng.normal(0.82, 0.1))
            elif kind == "forged_qr":
                r = dict(sig_valid=0, scan_count=1, regions=1, registered_sale=0, scratch_wrong=0, scratch_ok=0,
                         seal_broken=int(rng.random() < 0.3), strip_dev=abs(rng.normal(0.2, 0.12)), price_ratio=rng.normal(0.8, 0.12))
            else:
                r = dict(sig_valid=1, scan_count=1 + rng.poisson(1.5), regions=int(rng.integers(1, 3)),
                         registered_sale=int(rng.random() < 0.15), scratch_wrong=0, scratch_ok=int(rng.random() < 0.3),
                         seal_broken=int(rng.random() < 0.2), strip_dev=abs(rng.normal(0.14, 0.1)), price_ratio=rng.uniform(0.6, 0.95))
        r["kind"], r["is_suspicious"] = kind, int(kind != "genuine")
        rows.append(r)
    return pd.DataFrame(rows)

sim = simulate_scans()
sim.to_csv(f"{OUT}/simulated_scans.csv", index=False)
print(sim.is_suspicious.value_counts().rename({0: "genuine", 1: "suspicious"}).to_string())
sim.head()

is_suspicious
genuine       3372
suspicious     628


,sig_valid,scan_count,regions,registered_sale,scratch_wrong,scratch_ok,seal_broken,strip_dev,price_ratio,kind,is_suspicious
0,1,3,1,1,1,0,0,0.038976,1.029391,genuine,0
1,1,1,1,1,0,0,0,0.025061,0.922628,genuine,0
2,1,1,1,1,0,1,0,0.043065,0.997090,genuine,0
3,1,2,2,1,0,1,0,0.068374,1.053063,genuine,0
4,1,1,1,1,0,0,0,0.007357,1.051563,genuine,0


In [13]:
FEATURES = ["sig_valid", "scan_count", "regions", "registered_sale", "scratch_wrong", "scratch_ok",
            "seal_broken", "strip_dev", "price_ratio"]

def compare_models(df, features, target, title=""):
    X, y = df[features], df[target]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    models = {
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    }
    results, fitted = [], {}
    for name, m in models.items():
        if name == "Gradient Boosting":
            m.fit(Xtr, ytr, sample_weight=compute_sample_weight("balanced", ytr))   # same imbalance care as the forest
        else:
            m.fit(Xtr, ytr)
        proba = m.predict_proba(Xte)[:, 1]
        pred = (proba >= 0.5).astype(int)
        cv = cross_val_score(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1").mean()
        tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
        results.append({"model": name, "precision": precision_score(yte, pred), "recall": recall_score(yte, pred),
                        "f1": f1_score(yte, pred), "roc_auc": roc_auc_score(yte, proba), "cv_f1_mean": cv,
                        "missed_fakes": fn, "false_alarms": fp})
        fitted[name] = (m, Xte, yte)
    return pd.DataFrame(results).round(3), fitted

ml_results, fitted = compare_models(sim, FEATURES, "is_suspicious")
ml_results.to_csv(f"{OUT}/ml_comparison.csv", index=False)
print("Test-set comparison (simulated data):")
print(ml_results.to_string(index=False))

best = ml_results.sort_values(["recall", "f1"], ascending=False).iloc[0]["model"]
print(f"\nPicked by recall first, then F1: {best}")
imp = pd.Series(fitted[best][0].feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)
print("\nWhich signals the model leaned on:")
print(imp.to_string())
print("\nCaution: scores this high mean my simulated problem is EASY, not that real counterfeits are easy to catch.")
print("The importances mirror how I built the simulation (the 'diluted' fakes differ mostly in price and strip colour).")

Test-set comparison (simulated data):
            model  precision  recall    f1  roc_auc  cv_f1_mean  missed_fakes  false_alarms
    Random Forest      0.974   0.962 0.968    0.998       0.963             6             4
Gradient Boosting      0.927   0.975 0.950    0.999       0.962             4            12

Picked by recall first, then F1: Gradient Boosting

Which signals the model leaned on:
price_ratio        0.717
strip_dev          0.186
registered_sale    0.034
sig_valid          0.025
regions            0.012
scan_count         0.011
seal_broken        0.007
scratch_ok         0.007
scratch_wrong      0.001

Caution: scores this high mean my simulated problem is EASY, not that real counterfeits are easy to catch.
The importances mirror how I built the simulation (the 'diluted' fakes differ mostly in price and strip colour).


In [14]:
# Try different alert thresholds. Lower threshold = catches more fakes but raises more false alarms.
m, Xte, yte = fitted[best]
proba = m.predict_proba(Xte)[:, 1]
th = []
for t in (0.2, 0.3, 0.5, 0.7):
    pred = (proba >= t).astype(int)
    th.append({"threshold": t, "recall": recall_score(yte, pred), "precision": precision_score(yte, pred),
               "missed_fakes": int(((pred == 0) & (yte == 1)).sum()), "false_alarms": int(((pred == 1) & (yte == 0)).sum())})
pd.DataFrame(th).round(3)

,threshold,recall,precision,missed_fakes,false_alarms
0,0.2,0.994,0.867,1,24
1,0.3,0.987,0.906,2,16
2,0.5,0.975,0.927,4,12
3,0.7,0.968,0.962,5,6


### Using your own Kaggle file
Put `counterfeit_products_clean.csv` next to this notebook (in Colab: the folder icon on the left, then upload), set the three names below, and run the cell. It trains the same two models on your data.

**Check before trusting it:** the dataset is about counterfeit *products*, so it may not contain fertiliser-specific signals (strip colour, scratch code, scan history). The model can then only learn what the columns describe. Tell me the column names and I will map them onto the features here.

In [15]:
CSV_PATH = "counterfeit_products_clean.csv"
TARGET = "is_counterfeit"          # <- change to your label column
FEATURE_COLUMNS = None             # <- or a list of column names; None = use all other columns

if os.path.exists(CSV_PATH):
    user_df = pd.read_csv(CSV_PATH)
    print("Columns found:", list(user_df.columns))
    if TARGET not in user_df.columns:
        print(f"Set TARGET to one of the columns above (currently '{TARGET}').")
    else:
        cols = FEATURE_COLUMNS or [c for c in user_df.columns if c != TARGET]
        X = pd.get_dummies(user_df[cols], drop_first=True)          # turn text columns into 0/1 columns
        user_df2 = pd.concat([X, user_df[TARGET]], axis=1).dropna()
        user_results, _ = compare_models(user_df2, list(X.columns), TARGET)
        print(user_results.to_string(index=False))
else:
    print(f"'{CSV_PATH}' not found here, skipping. This is fine for the demo.")

Columns found: ['product_id', 'seller_id', 'category', 'brand', 'price', 'seller_rating', 'seller_reviews', 'product_images', 'description_length', 'shipping_time_days', 'spelling_errors', 'domain_age_days', 'contact_info_complete', 'return_policy_clear', 'payment_methods_count', 'listing_date', 'seller_country', 'shipping_origin', 'views', 'purchases', 'wishlist_adds', 'certification_badges', 'warranty_months', 'bulk_orders', 'unusual_payment_patterns', 'ip_location_mismatch', 'is_counterfeit', 'brand_canonical', 'brand_is_spoofed']
            model  precision  recall  f1  roc_auc  cv_f1_mean  missed_fakes  false_alarms
    Random Forest        1.0     1.0 1.0      1.0         1.0             0             0
Gradient Boosting        1.0     1.0 1.0      1.0         1.0             0             0


### Putting rules and the model together
Rules are the **safety floor**: a bad signature is `INVALID` no matter what the model says. The model only adds an opinion on the grey area, and can only make the verdict *more* cautious, never less.

In [17]:
def combined_verdict(rule_result, ml_prob):
    order = ["CONSISTENT", "UNVERIFIED_OFFLINE", "CAUTION", "SUSPICIOUS", "INVALID"]
    v = rule_result["verdict"]
    if v == "INVALID":
        return v
    ml_v = "SUSPICIOUS" if ml_prob >= 0.7 else "CAUTION" if ml_prob >= 0.4 else "CONSISTENT"
    return max(v, ml_v, key=order.index)

demo = pd.DataFrame([{"sig_valid": 1, "scan_count": 1, "regions": 1, "registered_sale": 1, "scratch_wrong": 0, "scratch_ok": 1,
                      "seal_broken": 0, "strip_dev": 0.05, "price_ratio": 1.0},
                     {"sig_valid": 1, "scan_count": 1, "regions": 1, "registered_sale": 1, "scratch_wrong": 0, "scratch_ok": 0,
                      "seal_broken": 1, "strip_dev": 0.30, "price_ratio": 0.78}])
for i, p_ in enumerate(fitted[best][0].predict_proba(demo)[:, 1]):
    print(f"example {i+1}: model risk={p_:.2f}  rules say CONSISTENT  ->  combined: {combined_verdict({'verdict': 'CONSISTENT'}, p_)}")

example 1: model risk=0.00  rules say CONSISTENT  ->  combined: CONSISTENT
example 2: model risk=1.00  rules say CONSISTENT  ->  combined: SUSPICIOUS


## Part 7: Benchmark against a conventional database (Objective 2)

**What:** write the same stream of supply-chain events into (a) the KrishiVerify ledger (hash + signature) and (b) the same table with crypto switched off, at growing sizes. We measure **latency, throughput, storage, CPU time**, and **communication size**.
**Why:** your Objective 2 asks for evidence that the extra security is affordable. The honest answer is a number: "tamper evidence costs X microseconds and Y bytes per event".

**Energy.** A notebook cannot read a power meter. We *estimate* energy as CPU-seconds × an assumed active power (`ASSUMED_WATTS`). For real numbers, run this on the Raspberry Pi / phone with a USB power meter and replace the estimate.

In [18]:
ASSUMED_WATTS = 3.0     # assumption for a Raspberry Pi 4 under load; entry-level Android is roughly 1 to 2 W

def bench_ledger(secure: bool, n_units: int):
    reg = Registry()
    for a in (mfr, dist, ret1):
        reg.register(a)
    path = os.path.join(tempfile.mkdtemp(), "bench.db")
    led = Ledger(reg, path=path, secure=secure)
    lat = []
    cpu0, t0 = time.process_time(), time.perf_counter()
    for i in range(n_units):
        uid = f"MFR-01:BENCH:{i:06d}"
        seq = [("MANUFACTURED", mfr, {"scratch_hash": "x" * 64}), ("SHIPPED", mfr, {"to": "D"}), ("RECEIVED", dist, {}),
               ("SHIPPED", dist, {"to": "R"}), ("RECEIVED", ret1, {}), ("SOLD", ret1, {})]
        for et, actor, body in seq:
            s = time.perf_counter()
            led.append(uid, et, actor, body)
            lat.append((time.perf_counter() - s) * 1e6)
    wall, cpu = time.perf_counter() - t0, time.process_time() - cpu0
    led.db.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    n_events = n_units * 6
    size = os.path.getsize(path)
    t1 = time.perf_counter()
    led.verify_chain() if secure else None
    audit_s = time.perf_counter() - t1 if secure else float("nan")
    return {"system": "KrishiVerify (hash+sign)" if secure else "Conventional DB", "units": n_units, "events": n_events,
            "p50_us": np.percentile(lat, 50), "p95_us": np.percentile(lat, 95), "events_per_s": n_events / wall,
            "bytes_per_event": size / n_events, "cpu_ms_per_event": cpu / n_events * 1000,
            "est_mJ_per_event": cpu / n_events * ASSUMED_WATTS * 1000, "full_audit_s": audit_s}

bench = pd.DataFrame([bench_ledger(sec, n) for n in (200, 1000, 3000) for sec in (False, True)]).round(2)
bench.to_csv(f"{OUT}/benchmark_ledger.csv", index=False)
print(bench.to_string(index=False))

                  system  units  events  p50_us  p95_us  events_per_s  bytes_per_event  cpu_ms_per_event  est_mJ_per_event  full_audit_s
         Conventional DB    200    1200   76.90  224.47       5942.88           201.39              0.10              0.31           NaN
KrishiVerify (hash+sign)    200    1200  124.80  267.35       5451.54           464.21              0.14              0.43          0.18
         Conventional DB   1000    6000   60.00  157.01       8506.08           174.08              0.08              0.25           NaN
KrishiVerify (hash+sign)   1000    6000  143.95  317.81       4721.49           438.27              0.18              0.55          1.00
         Conventional DB   3000   18000   70.10  192.80       5308.93           169.98              0.11              0.32           NaN
KrishiVerify (hash+sign)   3000   18000  124.40  297.40       3925.04           432.81              0.17              0.51          2.53


In [19]:
# Overhead of security as a ratio, at the largest size
big = bench[bench.units == bench.units.max()].set_index("system")
oh = (big.loc["KrishiVerify (hash+sign)", ["p50_us", "bytes_per_event", "cpu_ms_per_event"]] /
      big.loc["Conventional DB", ["p50_us", "bytes_per_event", "cpu_ms_per_event"]]).round(2)
print("Security overhead (KrishiVerify / Conventional):")
print(oh.to_string())

Security overhead (KrishiVerify / Conventional):
p50_us              1.77
bytes_per_event     2.55
cpu_ms_per_event    1.55


In [20]:
# Communication overhead: bytes a phone sends and receives for one online verification, and for a queued offline batch
req = json.dumps({"qr": qr, "scratch": scratch, "seal": "intact", "region": "Telangana"}).encode()
resp = json.dumps({"verdict": "CONSISTENT", "risk": 0, "flags": []}).encode()
queue = [json.dumps({"qr": qr, "ts": "2026-10-21T10:00:00Z", "region": "Telangana", "seal": "intact"}).encode() for _ in range(100)]
raw_q = sum(len(x) for x in queue)
gz_q = len(gzip.compress(b"\n".join(queue)))
comm = pd.DataFrame([
    {"item": "QR payload on pack", "bytes": len(qr)},
    {"item": "Online verify: request", "bytes": len(req)},
    {"item": "Online verify: response", "bytes": len(resp)},
    {"item": "Registry snapshot cached on phone", "bytes": len(json.dumps(registry.snapshot()))},
    {"item": "100 queued offline scans, raw", "bytes": raw_q},
    {"item": "100 queued offline scans, gzip", "bytes": gz_q},
])
comm.to_csv(f"{OUT}/communication_overhead.csv", index=False)
comm

,item,bytes
0,QR payload on pack,187
1,Online verify: request,261
2,Online verify: response,49
3,Registry snapshot cached on phone,574
4,"100 queued offline scans, raw",26800
5,"100 queued offline scans, gzip",401


## Part 8: Failures, edge vs centralised, and cost (Objective 2, steps 4 to 7)

**Failure injection.** We deliberately cause: duplicate requests, delayed or out-of-order messages, a device crash mid-write, and a network drop. The question for each is: *does the data stay correct?*

In [22]:
fail = []

# 1. duplicate requests (e.g. a retry after a timeout): same event_id sent 100 times
reg3, led3 = fresh_world()
u, _, _ = issue_unit(mfr, led3, "UREA-46-50KG", "B9", 1, "2026-10-01", "2028-10-01")
for _ in range(100):
    led3.append(u, "SHIPPED", mfr, {"to": "D"}, event_id="same-event-id")
rows_n = led3.db.execute("SELECT COUNT(*) FROM events WHERE unit_id=?", (u,)).fetchone()[0]
fail.append({"injected fault": "100 duplicate SHIPPED requests", "result": f"{rows_n - 1} SHIPPED row stored (expected 1)", "data correct": rows_n == 2})

# 2. delayed / out-of-order messages arriving after reconnection
reg4, led4 = fresh_world()
u, _, _ = issue_unit(mfr, led4, "UREA-46-50KG", "B9", 2, "2026-10-01", "2028-10-01")
pending = [("RECEIVED", dist, {}), ("SHIPPED", mfr, {"to": "D"}), ("SOLD", ret1, {}), ("RECEIVED", ret1, {}), ("SHIPPED", dist, {"to": "R"})]
naive_rejected = 0
for et, a, b in pending:                       # naive: apply in arrival order
    try:
        led4.append(u, et, a, b, event_id=f"naive-{et}-{a.actor_id}")
    except Rejected:
        naive_rejected += 1
reg5, led5 = fresh_world()
u5_, _, _ = issue_unit(mfr, led5, "UREA-46-50KG", "B9", 3, "2026-10-01", "2028-10-01")
queue_ = [(et, a, b, f"q-{i}") for i, (et, a, b) in enumerate(pending)]
progress = True
while queue_ and progress:                      # retry loop: keep applying whatever now fits
    progress, still = False, []
    for et, a, b, eid in queue_:
        try:
            led5.append(u5_, et, a, b, event_id=eid); progress = True
        except Rejected:
            still.append((et, a, b, eid))
    queue_ = still
final = led5.unit_history(u5_)[0][-1]["type"]
fail.append({"injected fault": "5 delayed messages in wrong order", "result": f"naive apply rejected {naive_rejected}; retry-queue left {len(queue_)} unapplied, final state {final}", "data correct": len(queue_) == 0 and final == "SOLD"})

# 3. device crash in the middle of a write
reg6, led6 = fresh_world()
u, _, _ = issue_unit(mfr, led6, "UREA-46-50KG", "B9", 4, "2026-10-01", "2028-10-01")
before = led6.db.execute("SELECT COUNT(*) FROM events").fetchone()[0]
try:
    led6.append(u, "SHIPPED", mfr, {"to": "D"}, _crash=True)
except RuntimeError:
    pass
after = led6.db.execute("SELECT COUNT(*) FROM events").fetchone()[0]
fail.append({"injected fault": "crash after INSERT, before COMMIT", "result": f"rows before={before}, after={after}; chain ok={led6.verify_chain()[0]}", "data correct": before == after and led6.verify_chain()[0]})

# 4. network disconnect: phone keeps verifying from cache and queues scans for later
cache = reg.snapshot()
offline_phone = Verifier(cache, ledger=None)
batch = [offline_phone.verify(q, online=False) for _ in range(50)]
queued = [{"qr": q, "ts": dt.datetime.now(dt.timezone.utc).isoformat(), "region": "Telangana"} for _ in batch]
ok_n = sum(b["verdict"] == "UNVERIFIED_OFFLINE" for b in batch)
for item in queued:                              # connectivity returns: flush the queue
    Verifier(reg.snapshot(), led).verify(item["qr"], online=True)
fail.append({"injected fault": "network down for 50 scans", "result": f"{ok_n}/50 answered offline (signature, recall list, expiry); {len(queued)} scans synced after reconnection", "data correct": ok_n == 50})

failures = pd.DataFrame(fail)
failures.to_csv(f"{OUT}/failure_injection.csv", index=False)
pd.set_option("display.max_colwidth", 120)
failures

,injected fault,result,data correct
0,100 duplicate SHIPPED requests,1 SHIPPED row stored (expected 1),True
1,5 delayed messages in wrong order,"naive apply rejected 2; retry-queue left 0 unapplied, final state SOLD",True
2,"crash after INSERT, before COMMIT","rows before=1, after=1; chain ok=True",True
3,network down for 50 scans,"50/50 answered offline (signature, recall list, expiry); 50 scans synced after reconnection",True


**Edge-assisted vs centralised verification.** *Centralised*: every scan is a round trip to the server. *Edge-assisted*: the phone does the signature, recall and expiry check locally and only the extra checks (scan history, scratch code) need the network.

The network profiles below are **assumed** typical values for 4G, weak 3G and EDGE rural coverage. They are inputs for a simulation, not measurements. Replace them with your field measurements.

In [23]:
def measure_local_verify_ms(n=300):
    v = Verifier(registry.snapshot(), None)
    t = time.perf_counter()
    for _ in range(n):
        v.verify(qr, online=False)
    return (time.perf_counter() - t) / n * 1000

local_ms = measure_local_verify_ms()
server_ms = bench[(bench.system == "KrishiVerify (hash+sign)")].p50_us.iloc[-1] / 1000 * 6 + local_ms   # ~ one pack's worth of ledger reads + verify
PROFILES = {"4G good": dict(rtt=60, loss=0.00), "3G weak": dict(rtt=300, loss=0.05), "EDGE rural": dict(rtt=900, loss=0.20), "No network": dict(rtt=None, loss=1.0)}

def simulate_central(rtt, loss, n=2000, timeout_ms=3000, retries=2, rng=np.random.default_rng(1)):
    if rtt is None:
        return [None] * n
    out = []
    for _ in range(n):
        t, done = 0.0, False
        for attempt in range(retries + 1):
            t += rtt * 3 + server_ms               # TLS handshake ~2 RTT + request ~1 RTT
            if rng.random() >= loss:
                done = True; break
            t += timeout_ms - rtt * 3              # waited for a reply that never came
        out.append(t if done else None)
    return out

rows_e = []
for name, p in PROFILES.items():
    c = simulate_central(**p)
    ok = [x for x in c if x is not None]
    rows_e.append({"network": name, "central success %": round(100 * len(ok) / len(c), 1),
                   "central p50 ms": round(np.percentile(ok, 50)) if ok else None,
                   "central p95 ms": round(np.percentile(ok, 95)) if ok else None,
                   "edge success % (signature, recall, expiry)": 100.0, "edge ms": round(local_ms, 2)})
edge_vs_central = pd.DataFrame(rows_e)
edge_vs_central.to_csv(f"{OUT}/edge_vs_central.csv", index=False)
edge_vs_central

,network,central success %,central p50 ms,central p95 ms,"edge success % (signature, recall, expiry)",edge ms
0,4G good,100.0,181.0,181.0,100.0,0.21
1,3G weak,100.0,901.0,3902.0,100.0,0.21
2,EDGE rural,98.8,2701.0,5702.0,100.0,0.21
3,No network,0.0,NaN,NaN,100.0,0.21


**Cost per product verified.** Costs are **placeholder assumptions in rupees**. Change them in the dictionary and re-run. The point is the *shape* of the answer: fixed server and maintenance costs get spread over more verifications as volume grows, and edge-assisted checks cut mobile-data cost.

In [24]:
ASSUME = {
    "server_inr_month": 1500,            # small cloud VM + backup (placeholder)
    "maintenance_inr_month": 8000,       # part-time admin / support (placeholder)
    "registry_ops_inr_month": 3000,      # key management, onboarding, audits (placeholder)
    "data_inr_per_mb": 0.10,             # farmer's mobile data (placeholder)
    "label_inr_per_unit": 0.25,          # printed QR + scratch panel, borne by manufacturer (placeholder)
}
bytes_central = len(req) + len(resp) + 2500        # + rough TLS handshake bytes
bytes_edge = (len(req) + len(resp)) * 0.3          # only some scans need the server (assumption: 30%)

def cost_per_verification(volume, bytes_per_verif):
    fixed = ASSUME["server_inr_month"] + ASSUME["maintenance_inr_month"] + ASSUME["registry_ops_inr_month"]
    data = bytes_per_verif / 1e6 * ASSUME["data_inr_per_mb"]
    return fixed / volume + data

cost = pd.DataFrame([{"verifications/month": v,
                      "centralised ₹/verification": round(cost_per_verification(v, bytes_central), 4),
                      "edge-assisted ₹/verification": round(cost_per_verification(v, bytes_edge), 4)}
                     for v in (10_000, 100_000, 1_000_000, 10_000_000)])
cost["label cost ₹/unit (manufacturer)"] = ASSUME["label_inr_per_unit"]
cost.to_csv(f"{OUT}/cost_model.csv", index=False)
cost

,verifications/month,centralised ₹/verification,edge-assisted ₹/verification,label cost ₹/unit (manufacturer)
0,10000,1.2503,1.2500,0.25
1,100000,0.1253,0.1250,0.25
2,1000000,0.0128,0.0125,0.25
3,10000000,0.0015,0.0013,0.25


## Part 9: QR readability under bad conditions (a lab proxy for field testing)

**What:** make QR codes for 40 packs, then damage the images the way real life does (blur, dim light, glare, tilt, low-resolution cameras, heavy JPEG compression) and count how often a standard decoder still reads them. We also compare **error-correction levels** L, M, Q, H.
**Why:** your Objective 1 asks for validation across devices and conditions. This gives a first, repeatable number *before* you spend time in the field. It does **not** replace real trials with real phones, people and villages.

Error correction is a trade-off: higher levels survive more damage but make the QR denser, which is harder for cheap cameras.

**Read the table carefully.** The decoder here is OpenCV's, and the damage is simulated. Phone scanners (Google ML Kit, ZXing) behave differently, so use the table to *compare conditions and error-correction levels with each other*, not as the success rate you will get in a village. The real number comes from Part 10's field trials.

In [25]:
EC = {"L": cv2.QRCodeEncoder_CORRECT_LEVEL_L, "M": cv2.QRCodeEncoder_CORRECT_LEVEL_M,
      "Q": cv2.QRCodeEncoder_CORRECT_LEVEL_Q, "H": cv2.QRCodeEncoder_CORRECT_LEVEL_H}

def make_qr_image(text, level="M", scale=6):
    p = cv2.QRCodeEncoder_Params(); p.correction_level = EC[level]
    img = cv2.QRCodeEncoder.create(p).encode(text)
    img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_NEAREST)
    return cv2.copyMakeBorder(img, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255)

def degrade(img, kind, rng):
    h, w = img.shape
    if kind == "clean":       return img
    if kind == "blur":        return cv2.GaussianBlur(img, (0, 0), 2.2)
    if kind == "heavy_blur":  return cv2.GaussianBlur(img, (0, 0), 3.6)
    if kind == "low_light":
        return np.clip(img * 0.35 + rng.normal(0, 14, img.shape), 0, 255).astype(np.uint8)
    if kind == "glare":
        yy, xx = np.mgrid[0:h, 0:w]
        cx, cy = rng.integers(w // 4, 3 * w // 4), rng.integers(h // 4, 3 * h // 4)
        blob = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * (0.18 * w) ** 2)))
        return np.clip(img + blob * 255, 0, 255).astype(np.uint8)
    if kind == "rotate_25":
        pad = int(0.25 * max(h, w))          # pad first so rotation does not crop the QR corners
        img = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=255)
        h, w = img.shape
        M = cv2.getRotationMatrix2D((w / 2, h / 2), 25, 1.0)
        return cv2.warpAffine(img, M, (w, h), borderValue=255)
    if kind == "tilt":
        src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
        dst = np.float32([[0.08 * w, 0.05 * h], [0.95 * w, 0], [w, h], [0, 0.92 * h]])
        return cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (w, h), borderValue=255)
    if kind == "low_res_cam":
        small = cv2.resize(img, (w // 3, h // 3), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if kind == "jpeg_q15":
        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 15])
        return cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
    raise ValueError(kind)

CONDITIONS = ["clean", "blur", "heavy_blur", "low_light", "glare", "rotate_25", "tilt", "low_res_cam", "jpeg_q15"]
# OpenCV's ArUco-based QR detector is more reliable than the default one; fall back if your OpenCV is older.
detector = cv2.QRCodeDetectorAruco() if hasattr(cv2, "QRCodeDetectorAruco") else cv2.QRCodeDetector()
print("Decoder:", type(detector).__name__, "| OpenCV", cv2.__version__)
rng = np.random.default_rng(3)

# 40 real signed QR strings from the same issuing code as the notebook
test_qrs = [issue_unit(mfr, Ledger(registry), "UREA-46-50KG", "B2610A", 100 + i, "2026-10-01", "2028-10-01")[1] for i in range(40)]

res_rows = []
for level in EC:
    for cond in CONDITIONS:
        ok = 0
        for t in test_qrs:
            im = degrade(make_qr_image(t, level), cond, rng)
            try:
                txt, _, _ = detector.detectAndDecode(im)
            except cv2.error:
                txt = ""
            ok += int(txt == t)
        res_rows.append({"EC level": level, "condition": cond, "decode success %": round(100 * ok / len(test_qrs))})

qr_table = pd.DataFrame(res_rows).pivot(index="condition", columns="EC level", values="decode success %").loc[CONDITIONS]
qr_table.to_csv(f"{OUT}/qr_robustness.csv")
side = make_qr_image(test_qrs[0], "M")
print(f"QR modules per side at level M for this payload: {(cv2.QRCodeEncoder.create().encode(test_qrs[0]).shape[0])}")
qr_table

Decoder: QRCodeDetectorAruco | OpenCV 5.0.0
QR modules per side at level M for this payload: 53


EC level,H,L,M,Q
condition,,,,
clean,100,100,100,100
blur,50,5,5,60
heavy_blur,0,0,0,0
low_light,88,98,90,95
glare,100,100,100,100
rotate_25,100,100,100,100
tilt,100,100,100,100
low_res_cam,100,100,100,100
jpeg_q15,100,100,100,100


## Part 10: Field-trial recording sheet

The simulations above are only a first pass. This template is the table you fill in during real trials. One row per scan attempt. It covers the "diverse users, regions, devices, products, conditions" requirement of Objective 1 and the cost and performance inputs of Objective 2.

In [26]:
template_cols = ["trial_id", "date", "district", "village_or_shop", "user_group (farmer/retailer/distributor/officer)",
                 "language", "phone_model", "android_version", "camera_mp", "network (none/2G/3G/4G)", "light (bright/shade/dim)",
                 "product_sku", "batch", "pack_type (genuine-known/suspect/lab-tested)", "qr_read_ok (y/n)", "seconds_to_result",
                 "attempts_needed", "mode (offline/online)", "verdict_shown", "ground_truth", "seal_state", "strip_used (y/n)",
                 "user_understood_message (1-5)", "needed_help (y/n)", "notes"]
pd.DataFrame(columns=template_cols).to_csv(f"{OUT}/field_trial_template.csv", index=False)
print(f"Wrote {OUT}/field_trial_template.csv with {len(template_cols)} columns")
print("\nFiles produced in", OUT + ":")
for f in sorted(os.listdir(OUT)):
    print("  ", f)

Wrote krishiverify_outputs/field_trial_template.csv with 25 columns

Files produced in krishiverify_outputs:
   attack_validation.csv
   benchmark_ledger.csv
   communication_overhead.csv
   cost_model.csv
   edge_vs_central.csv
   failure_injection.csv
   field_trial_template.csv
   ml_comparison.csv
   qr_robustness.csv
   simulated_scans.csv


## What to do next
1. **Upload your `counterfeit_products_clean.csv`** and run the "Using your own Kaggle file" cell. Share the column names and I will map them onto the risk features.
2. **Get real strip colours.** The strip reference values here are placeholders; they need real strips and lab-tested samples.
3. **Run Parts 7 to 9 on the target hardware** (entry-level Android or Raspberry Pi) and replace the estimated energy and network numbers with measured ones.
4. **Read the design and threat-model document** for the governance model (Objective 3), the user-workflow and security measures (Objective 4), and the field-trial protocol.